In [1]:
from pathlib import Path
import json
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
from ultralytics import YOLO

from torchvision import datasets, transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

PROJECT_ROOT = Path.cwd().parent

PROCESSED_DATA = (
    PROJECT_ROOT
    / "datasets"
    / "processed"
)

EXPERIMENTS_DIR = (
    PROJECT_ROOT
    / "experiments"
)

MODELS_DIR = (
    PROJECT_ROOT
    / "models"
)

FINAL_TEST_DIR = (
    EXPERIMENTS_DIR
    / "final_testing"
)

FINAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)

DEVICE = (
    0
    if torch.cuda.is_available()
    else "cpu"
)

print("Inspectra — Final Testing")
print("=" * 60)
print("Device:", DEVICE)

Inspectra — Final Testing
Device: 0


In [2]:
models = {
    "bottle": {
        "baseline": (
            EXPERIMENTS_DIR
            / "bottle"
            / "baseline"
            / "baseline"
            / "weights"
            / "best.pt"
        ),
        "finetuned": (
            EXPERIMENTS_DIR
            / "finetuning"
            / "bottle"
            / "finetuned"
            / "weights"
            / "best.pt"
        ),
    },

    "pcb": {
        "baseline": (
            EXPERIMENTS_DIR
            / "pcb"
            / "baseline"
            / "baseline"
            / "weights"
            / "best.pt"
        ),
        "finetuned": (
            EXPERIMENTS_DIR
            / "finetuning"
            / "pcb"
            / "finetuned"
            / "weights"
            / "best.pt"
        ),
    },

    "road": {
        "baseline": (
            MODELS_DIR
            / "road"
            / "baseline_resnet18.pt"
        ),
        "finetuned": (
            MODELS_DIR
            / "road"
            / "finetuned"
            / "resnet18_finetuned.pt"
        ),
    },
}

for dataset, variants in models.items():

    print(f"\n{dataset.upper()}")

    for variant, path in variants.items():

        print(
            f"{variant:10}: "
            f"{'FOUND' if path.exists() else 'MISSING'}"
        )


BOTTLE
baseline  : FOUND
finetuned : FOUND

PCB
baseline  : FOUND
finetuned : FOUND

ROAD
baseline  : FOUND
finetuned : FOUND


In [3]:
def final_test_yolo(
    dataset_name,
    variant,
    model_path
):

    if not model_path.exists():

        print(
            f"{dataset_name} {variant}: "
            "model not found"
        )

        return None

    data_yaml = (
        PROCESSED_DATA
        / dataset_name
        / "data.yaml"
    )

    model = YOLO(
        str(model_path)
    )

    output_name = (
        f"{dataset_name}_{variant}"
    )

    start = time.perf_counter()

    metrics = model.val(
        data=str(data_yaml),
        split="test",
        imgsz=640,
        batch=4,
        device=DEVICE,
        plots=True,
        project=str(
            FINAL_TEST_DIR
        ),
        name=output_name,
        exist_ok=True,
        verbose=True,
    )

    elapsed = (
        time.perf_counter()
        - start
    )

    result = {
        "dataset": dataset_name,
        "variant": variant,
        "precision": float(
            metrics.box.mp
        ),
        "recall": float(
            metrics.box.mr
        ),
        "map50": float(
            metrics.box.map50
        ),
        "map50_95": float(
            metrics.box.map
        ),
        "evaluation_seconds": elapsed,
    }

    return result

In [4]:
bottle_final = []

for variant in [
    "baseline",
    "finetuned",
]:

    result = final_test_yolo(
        "bottle",
        variant,
        models["bottle"][variant]
    )

    if result is not None:
        bottle_final.append(
            result
        )

bottle_final

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.40.1 ms, read: 5.50.7 MB/s, size: 39.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\bottle\test\labels.cache... 920 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 920/920  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 230/230 18.7it/s 12.3s0.1s
                   all        920       1829      0.967      0.971      0.982      0.896
                   Cap        468        473       0.97      0.967      0.987      0.879
               Missing        348        441      0.996      0.998      0.993       0.85
          Wrong bottle        430 

[{'dataset': 'bottle',
  'variant': 'baseline',
  'precision': 0.9670109594891115,
  'recall': 0.9709702968514925,
  'map50': 0.9820735036274845,
  'map50_95': 0.8963879891309094,
  'evaluation_seconds': 30.68284599998151},
 {'dataset': 'bottle',
  'variant': 'finetuned',
  'precision': 0.9737279159862814,
  'recall': 0.9652940142357527,
  'map50': 0.9790346385813005,
  'map50_95': 0.8966904633426118,
  'evaluation_seconds': 29.69574540000758}]

In [5]:
pcb_final = []

for variant in [
    "baseline",
    "finetuned",
]:

    result = final_test_yolo(
        "pcb",
        variant,
        models["pcb"][variant]
    )

    if result is not None:
        pcb_final.append(
            result
        )

pcb_final

Ultralytics 8.4.117  Python-3.11.5 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 6GB Laptop GPU, 6144MiB)
Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.20.2 ms, read: 15.33.6 MB/s, size: 96.7 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips/
val: Scanning D:\Inspectra\datasets\processed\pcb\test\labels.cache... 829 images, 239 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1068/1068  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 267/267 20.4it/s 13.1s0.1s
                   all       1068       1662       0.98      0.984      0.988      0.555
            mouse_bite        131        262      0.977      0.985      0.988      0.541
                  spur        138        279      0.985      0.973      0.988      0.525
          missing_hole        14

[{'dataset': 'pcb',
  'variant': 'baseline',
  'precision': 0.9795733243367813,
  'recall': 0.9839463915129786,
  'map50': 0.9879771306637831,
  'map50_95': 0.5548077556115211,
  'evaluation_seconds': 30.141462800005684},
 {'dataset': 'pcb',
  'variant': 'finetuned',
  'precision': 0.978857361572334,
  'recall': 0.987615852637089,
  'map50': 0.9878409828808247,
  'map50_95': 0.5735049068611203,
  'evaluation_seconds': 26.897960300004343}]

In [6]:
detection_final_results = (
    bottle_final
    + pcb_final
)

detection_final_df = pd.DataFrame(
    detection_final_results
)

display(
    detection_final_df
)

,dataset,variant,precision,recall,map50,map50_95,evaluation_seconds
0,bottle,baseline,0.967011,0.970970,0.982074,0.896388,30.682846
1,bottle,finetuned,0.973728,0.965294,0.979035,0.896690,29.695745
2,pcb,baseline,0.979573,0.983946,0.987977,0.554808,30.141463
3,pcb,finetuned,0.978857,0.987616,0.987841,0.573505,26.897960


In [7]:
comparison_rows = []

for dataset in [
    "bottle",
    "pcb",
]:

    dataset_results = (
        detection_final_df[
            detection_final_df[
                "dataset"
            ] == dataset
        ]
    )

    baseline = dataset_results[
        dataset_results[
            "variant"
        ] == "baseline"
    ]

    finetuned = dataset_results[
        dataset_results[
            "variant"
        ] == "finetuned"
    ]

    if (
        baseline.empty
        or finetuned.empty
    ):
        continue

    baseline = baseline.iloc[0]
    finetuned = finetuned.iloc[0]

    comparison_rows.append({
        "dataset": dataset,

        "baseline_map50_95":
            baseline["map50_95"],

        "finetuned_map50_95":
            finetuned["map50_95"],

        "map50_95_change":
            finetuned["map50_95"]
            - baseline["map50_95"],

        "baseline_precision":
            baseline["precision"],

        "finetuned_precision":
            finetuned["precision"],

        "baseline_recall":
            baseline["recall"],

        "finetuned_recall":
            finetuned["recall"],
    })

detection_comparison = pd.DataFrame(
    comparison_rows
)

display(
    detection_comparison
)

,dataset,baseline_map50_95,finetuned_map50_95,map50_95_change,baseline_precision,finetuned_precision,baseline_recall,finetuned_recall
0,bottle,0.896388,0.896690,0.000302,0.967011,0.973728,0.970970,0.965294
1,pcb,0.554808,0.573505,0.018697,0.979573,0.978857,0.983946,0.987616


In [8]:
if not detection_final_df.empty:

    pivot = detection_final_df.pivot(
        index="dataset",
        columns="variant",
        values="map50_95"
    )

    pivot.plot(
        kind="bar",
        figsize=(10, 6)
    )

    plt.ylabel(
        "mAP50-95"
    )

    plt.xlabel(
        "Dataset"
    )

    plt.title(
        "Baseline vs Fine-Tuned Detection"
    )

    plt.xticks(
        rotation=0
    )

    plt.grid(
        axis="y",
        alpha=0.3
    )

    plt.show()

<Figure size 1000x600 with 1 Axes>

In [9]:
road_device = (
    f"cuda:{DEVICE}"
    if isinstance(DEVICE, int)
    else DEVICE
)

road_transform = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[
            0.485,
            0.456,
            0.406
        ],
        std=[
            0.229,
            0.224,
            0.225
        ]
    ),
])

road_test = datasets.ImageFolder(
    PROCESSED_DATA
    / "road"
    / "test",
    transform=road_transform
)

road_test_loader = DataLoader(
    road_test,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=torch.cuda.is_available(),
)

print(
    "Classes:",
    road_test.classes
)

print(
    "Test images:",
    len(road_test)
)

Classes: ['Negative', 'Positive']
Test images: 6000


In [10]:
def load_road_model(
    path
):

    model = resnet18(
        weights=None
    )

    model.fc = torch.nn.Linear(
        model.fc.in_features,
        len(road_test.classes)
    )

    state = torch.load(
        path,
        map_location=road_device
    )

    model.load_state_dict(
        state
    )

    model = model.to(
        road_device
    )

    model.eval()

    return model

In [11]:
def evaluate_road_model(
    model
):

    predictions = []
    targets = []

    with torch.no_grad():

        for images, labels in road_test_loader:

            images = images.to(
                road_device
            )

            outputs = model(
                images
            )

            preds = (
                outputs
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            predictions.extend(
                preds
            )

            targets.extend(
                labels.numpy()
            )

    predictions = np.array(
        predictions
    )

    targets = np.array(
        targets
    )

    return {
        "accuracy": accuracy_score(
            targets,
            predictions
        ),

        "precision": precision_score(
            targets,
            predictions,
            average="binary"
        ),

        "recall": recall_score(
            targets,
            predictions,
            average="binary"
        ),

        "f1": f1_score(
            targets,
            predictions,
            average="binary"
        ),

        "predictions": predictions,

        "targets": targets,
    }

In [12]:
road_results = []

road_predictions = {}

for variant in [
    "baseline",
    "finetuned",
]:

    model = load_road_model(
        models["road"][variant]
    )

    result = evaluate_road_model(
        model
    )

    road_predictions[
        variant
    ] = result

    road_results.append({
        "dataset": "road",
        "variant": variant,
        "accuracy": result[
            "accuracy"
        ],
        "precision": result[
            "precision"
        ],
        "recall": result[
            "recall"
        ],
        "f1": result[
            "f1"
        ],
    })

road_results_df = pd.DataFrame(
    road_results
)

display(
    road_results_df
)

C:\Users\Garvit\AppData\Local\Temp\ipykernel_19444\178323199.py:14: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state = torch.load(
C:\Users\Garvit\AppData\Local\Temp\ipyk

,dataset,variant,accuracy,precision,recall,f1
0,road,baseline,0.999000,1.000000,0.998000,0.998999
1,road,finetuned,0.999333,0.999001,0.999667,0.999334


In [13]:
road_baseline = road_results_df[
    road_results_df[
        "variant"
    ] == "baseline"
].iloc[0]

road_finetuned = road_results_df[
    road_results_df[
        "variant"
    ] == "finetuned"
].iloc[0]

road_comparison = {
    "accuracy_change":
        road_finetuned["accuracy"]
        - road_baseline["accuracy"],

    "precision_change":
        road_finetuned["precision"]
        - road_baseline["precision"],

    "recall_change":
        road_finetuned["recall"]
        - road_baseline["recall"],

    "f1_change":
        road_finetuned["f1"]
        - road_baseline["f1"],
}

road_comparison

{'accuracy_change': np.float64(0.0003333333333332966),
 'precision_change': np.float64(-0.0009993337774816258),
 'recall_change': np.float64(0.0016666666666667052),
 'f1_change': np.float64(0.0003345564825071312)}

In [14]:
for variant in [
    "baseline",
    "finetuned",
]:

    result = road_predictions[
        variant
    ]

    cm = confusion_matrix(
        result["targets"],
        result["predictions"]
    )

    plt.figure(
        figsize=(6, 5)
    )

    plt.imshow(
        cm
    )

    plt.xticks(
        range(len(road_test.classes)),
        road_test.classes
    )

    plt.yticks(
        range(len(road_test.classes)),
        road_test.classes
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "Actual"
    )

    plt.title(
        f"Road — {variant}"
    )

    for i in range(
        cm.shape[0]
    ):

        for j in range(
            cm.shape[1]
        ):

            plt.text(
                j,
                i,
                cm[i, j],
                ha="center",
                va="center"
            )

    plt.colorbar()

    plt.show()

<Figure size 600x500 with 2 Axes>

<Figure size 600x500 with 2 Axes>

In [15]:
final_summary = []

for result in detection_final_results:

    final_summary.append({
        "dataset": result["dataset"],
        "task": "detection",
        "variant": result["variant"],
        "primary_metric": "mAP50-95",
        "score": result["map50_95"],
        "precision": result["precision"],
        "recall": result["recall"],
        "secondary_metric": result["map50"],
    })

for result in road_results:

    final_summary.append({
        "dataset": "road",
        "task": "classification",
        "variant": result["variant"],
        "primary_metric": "F1",
        "score": result["f1"],
        "precision": result["precision"],
        "recall": result["recall"],
        "secondary_metric": result["accuracy"],
    })

final_summary_df = pd.DataFrame(
    final_summary
)

display(
    final_summary_df
)

,dataset,task,variant,primary_metric,score,precision,recall,secondary_metric
0,bottle,detection,baseline,mAP50-95,0.896388,0.967011,0.970970,0.982074
1,bottle,detection,finetuned,mAP50-95,0.896690,0.973728,0.965294,0.979035
2,pcb,detection,baseline,mAP50-95,0.554808,0.979573,0.983946,0.987977
3,pcb,detection,finetuned,mAP50-95,0.573505,0.978857,0.987616,0.987841
4,road,classification,baseline,F1,0.998999,1.000000,0.998000,0.999000
5,road,classification,finetuned,F1,0.999334,0.999001,0.999667,0.999333


In [16]:
final_summary_df.to_csv(
    FINAL_TEST_DIR
    / "final_results.csv",
    index=False
)

with open(
    FINAL_TEST_DIR
    / "final_results.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_summary,
        file,
        indent=4
    )

print(
    "Final test results saved to:"
)

print(
    FINAL_TEST_DIR
)

Final test results saved to:
d:\Inspectra\experiments\final_testing


In [17]:
deployment_candidates = []

for dataset in [
    "bottle",
    "pcb",
]:

    rows = final_summary_df[
        (
            final_summary_df[
                "dataset"
            ] == dataset
        )
        &
        (
            final_summary_df[
                "task"
            ] == "detection"
        )
    ]

    if rows.empty:
        continue

    best = rows.loc[
        rows["score"].idxmax()
    ]

    deployment_candidates.append({
        "dataset": dataset,
        "selected_variant":
            best["variant"],
        "metric":
            best["primary_metric"],
        "score":
            best["score"],
    })

road_rows = final_summary_df[
    final_summary_df[
        "dataset"
    ] == "road"
]

if not road_rows.empty:

    best = road_rows.loc[
        road_rows["score"].idxmax()
    ]

    deployment_candidates.append({
        "dataset": "road",
        "selected_variant":
            best["variant"],
        "metric":
            best["primary_metric"],
        "score":
            best["score"],
    })

deployment_df = pd.DataFrame(
    deployment_candidates
)

display(
    deployment_df
)

deployment_df.to_csv(
    FINAL_TEST_DIR
    / "deployment_candidates.csv",
    index=False
)

,dataset,selected_variant,metric,score
0,bottle,finetuned,mAP50-95,0.896690
1,pcb,finetuned,mAP50-95,0.573505
2,road,finetuned,F1,0.999334
